# DS-MambaGCN on Google Colab (T4 / L4 / A100)

Run the cells from top to bottom. After a disconnect, open the notebook again and run cells **1 → 2 → 6**: finished runs are skipped and an interrupted run resumes from its last epoch.

Layout on Google Drive (`MyDrive/DS-MambaGCN_runs/`):
- `code/DS-MambaGCN/` this repository (cell 3 clones or updates it)
- `data/ntu_compact/` preprocessed NTU RGB+D 60 (`data.npy`, `meta.npz`), see the appendix
- `wheels/sm_XX/` Mamba CUDA kernels cached per GPU type (created automatically)
- `runs/` checkpoints, logs and `results.json` of every run (created automatically)
- `queue.log` experiment-queue log

## 1. Check the GPU
Runtime → Change runtime type → choose the GPU. The cell below must print it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc; free -g | head -2

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
D = '/content/drive/MyDrive/DS-MambaGCN_runs'
os.environ['D'] = D
os.makedirs(D, exist_ok=True)
print(os.listdir(D))

## 3. Get or update the code
For a **private** repository, first add a GitHub token as a Colab secret: key icon in the left bar → *Add new secret* → name `GITHUB_TOKEN`, value = a fine-grained token with *Contents: read* on this repository → enable *Notebook access*. Only the code folder is touched; `runs/` and `data/` are kept.

In [ ]:
REPO = 'https://github.com/nvtthai85/DS-MambaGCN.git'
try:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
except Exception:
    tok = None
url = REPO.replace('https://', f'https://x-access-token:{tok}@') if tok else REPO
code_dir = f'{D}/code/DS-MambaGCN'
if os.path.isdir(code_dir) and not os.path.isdir(f'{code_dir}/.git'):
    os.rename(code_dir, code_dir + '_old')          # earlier copy that was not a git clone
    print('moved the old code folder to', code_dir + '_old')
if os.path.isdir(f'{code_dir}/.git'):
    !git -C "{code_dir}" remote set-url origin "{url}" && git -C "{code_dir}" pull --ff-only
else:
    !mkdir -p "$D/code" && git clone -q "{url}" "{code_dir}"
!git -C "{code_dir}" remote set-url origin "{REPO}"
!git -C "{code_dir}" log --oneline -1

## 4. Check the NTU RGB+D 60 data
`data/ntu_compact/data.npy` and `meta.npz` must exist; otherwise run the appendix cell.

In [ ]:
!ls -la "$D/data/ntu_compact" 2>/dev/null || echo "MISSING: run the appendix cell"
!python -c "import json;r=json.load(open('$D/data/ntu_compact/report.json'));print(r['samples_kept'],'samples',r['classes'],'classes')" 2>/dev/null

## 5. Install the Mamba kernels for this GPU and copy code/data to local disk
First time on a GPU type: tries the upstream prebuilt wheel (about 1 min), otherwise compiles for this GPU only (15–25 min) and caches the result on Drive.

In [ ]:
!START=0 bash "$D/code/DS-MambaGCN/scripts/colab_session.sh"

### 5b. Tests and speed measurement
`run_120ep_h_est` is the estimated duration of one 120-epoch NTU60 X-Sub run on this GPU. Use it to choose the family and seeds in cell 6.

In [ ]:
%cd /content/DS-MambaGCN
!python -m pytest -q tests/ 2>&1 | tail -2
!python tools/bench_speed.py --config configs/ntu60_xsub/full.yaml --batch-size 16

## 6. Start the experiment queue (background)
Defaults follow the GPU memory (A100 40 GB: batch 32, no accumulation, two runs in parallel; T4: batch 32 as 16 × 2). Override with environment variables, e.g.
- `SEEDS=0` only the first pass (seed 0 of every configuration),
- `FAMILY=ntu60_xview` another benchmark (add `CONFIGS=full`),
- `PARALLEL=1` number of runs side by side.

Do **not** run the queue in two Colab sessions at the same time.

In [ ]:
!SEEDS=0,1,2,3,4 FAMILY=ntu60_xsub bash "$D/code/DS-MambaGCN/scripts/colab_session.sh"

## 7. Progress (run any time)

In [ ]:
FAMILY = 'ntu60_xsub_lite'      # the family you started in cell 6
SEEDS = '0'                      # the seeds you started in cell 6
import glob, os, subprocess
%cd /content
print(subprocess.run(f'tail -n 8 "{D}/queue.log"', shell=True, capture_output=True, text=True).stdout)
print(subprocess.run(f'cd "{D}/code/DS-MambaGCN" && python tools/run_queue.py --runs "{D}/runs" --data x '
                     f'--family {FAMILY} --seeds {SEEDS} --status', shell=True, capture_output=True, text=True).stdout)
for f in sorted(glob.glob(f'{D}/runs/{FAMILY}_*/seed*/stdout.log'), key=os.path.getmtime)[-4:]:
    last = open(f).read().strip().splitlines()[-1:] or ['(empty)']
    print(f.split('/runs/')[1].replace('/stdout.log', ''), '|', last[0][:150])
!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv

## 8. Aggregate results
Mean ± std over seeds, paired difference to the full model with a paired t-test, LaTeX rows.

In [ ]:
!cd /content && cd "$D/code/DS-MambaGCN" && python tools/aggregate_results.py --runs "$D/runs" --reference ntu60_xsub_lite_full --json-out "$D/summary.json"

## 9. Stop the queue
An interrupted run resumes from its last epoch the next time cell 6 runs.

In [ ]:
!pkill -f tools/run_queue.py; pkill -f train.py; sleep 3; pgrep -fa train.py || echo stopped

## Appendix: preprocess NTU RGB+D 60 from the raw .skeleton files
Put `nturgbd_skeletons_s001_to_s017.zip` and `NTU_RGBD_samples_with_missing_skeletons.txt` in `MyDrive/DS-MambaGCN_runs/raw/`.

In [ ]:
!mkdir -p /content/ntu_raw && unzip -q -o "$D/raw/nturgbd_skeletons_s001_to_s017.zip" -d /content/ntu_raw
!cd "$D/code/DS-MambaGCN" && python tools/ntu_preprocess.py --raw /content/ntu_raw --out /content/data/ntu_compact --max-action 60 \
    --missing "$D/raw/NTU_RGBD_samples_with_missing_skeletons.txt" --workers $(nproc)
!mkdir -p "$D/data" && cp -r /content/data/ntu_compact "$D/data/"